# Plant Disease Detection (MobileNetV2)

Train MobileNetV2 on your local `plantvillage dataset` and save:
- `models/plant_disease_mobilenetv2.keras`
- `models/class_names.json` (label ordering for Flask inference)


In [ ]:
from pathlib import Path
import json
import numpy as np

import tensorflow as tf
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "plantvillage dataset"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / "plant_disease_mobilenetv2.keras"
CLASS_NAMES_PATH = MODELS_DIR / "class_names.json"

# Always use original PlantVillage color images (not segmented/grayscale).
DATA_SUBDIR = "color"
# If True, apply OpenCV pipeline like Flask; if False, TF resize + normalize only.
USE_OPENCV_PREPROCESSING = True

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

print("Project root:", PROJECT_ROOT)
print("Data root:", DATA_ROOT)
print("DATA_SUBDIR:", DATA_SUBDIR)
print("USE_OPENCV_PREPROCESSING:", USE_OPENCV_PREPROCESSING)


In [ ]:
train_root = DATA_ROOT / DATA_SUBDIR

if not train_root.exists():
    raise FileNotFoundError(f"Dataset folder not found: {train_root}")

# Collect class folders (alphabetical) so label ordering matches inference.
class_names = sorted([p.name for p in train_root.iterdir() if p.is_dir()])
num_classes = len(class_names)

print("Num classes:", num_classes)
print("First 5 classes:", class_names[:5])

# Save class names so Flask uses the same ordering.
CLASS_NAMES_PATH.write_text(json.dumps(class_names, indent=2), encoding="utf-8")
print("Saved class names:", CLASS_NAMES_PATH)


In [ ]:
# Build (filepath, label) lists.
exts = ["*.jpg", "*.jpeg", "*.png", "*.webp"]
filepaths = []
labels = []

for class_idx, class_name in enumerate(class_names):
    class_dir = train_root / class_name
    for ext in exts:
        for fp in class_dir.glob(ext):
            filepaths.append(str(fp))
            labels.append(class_idx)

filepaths = np.array(filepaths)
labels = np.array(labels)
print("Total images:", len(filepaths))

# Train/val/test split (stratified)
test_size = 0.15
val_size = 0.15

train_files, test_files, train_labels, test_labels = train_test_split(
    filepaths,
    labels,
    test_size=test_size,
    stratify=labels,
    random_state=SEED,
)

val_ratio = val_size / (1.0 - test_size)
train_files, val_files, train_labels, val_labels = train_test_split(
    train_files,
    train_labels,
    test_size=val_ratio,
    stratify=train_labels,
    random_state=SEED,
)

print("Train/Val/Test:", len(train_files), len(val_files), len(test_files))


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
IMG_SIZE_TF = IMG_SIZE

if USE_OPENCV_PREPROCESSING:
    import cv2
    import sys
    sys.path.append(str(PROJECT_ROOT))
    from src.preprocessing.pipeline import LeafPreprocessor

    preprocessor = LeafPreprocessor()

    def cv2_preprocess_rgb(img_rgb_uint8):
        # img_rgb_uint8 comes in as RGB uint8.
        img_bgr = cv2.cvtColor(img_rgb_uint8, cv2.COLOR_RGB2BGR)
        out_rgb_uint8 = preprocessor.preprocess(img_bgr)  # uint8 RGB
        return out_rgb_uint8.astype(np.float32) / 255.0

    def load_and_preprocess(path, label):
        img_bytes = tf.io.read_file(path)
        img = tf.image.decode_image(img_bytes, channels=3, expand_animations=False)
        img = tf.cast(img, tf.uint8)

        out = tf.numpy_function(cv2_preprocess_rgb, [img], tf.float32)
        out.set_shape((IMG_SIZE_TF[1], IMG_SIZE_TF[0], 3))
        return out, tf.cast(label, tf.int32)

else:
    def load_and_preprocess(path, label):
        img_bytes = tf.io.read_file(path)
        img = tf.image.decode_image(img_bytes, channels=3, expand_animations=False)
        img = tf.image.resize(img, IMG_SIZE_TF)
        img = tf.image.convert_image_dtype(img, tf.float32)  # 0..1
        return img, tf.cast(label, tf.int32)

def make_ds(files, lbls, training: bool):
    ds = tf.data.Dataset.from_tensor_slices((files, lbls))
    if training:
        ds = ds.shuffle(buffer_size=min(len(files), 2000), seed=SEED, reshuffle_each_iteration=True)

    ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)

    if training:
        aug = tf.keras.Sequential(
            [
                tf.keras.layers.RandomFlip("horizontal"),
                tf.keras.layers.RandomRotation(0.12),
                tf.keras.layers.RandomZoom(0.12),
            ]
        )
        ds = ds.map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=AUTOTUNE)

    if training:
        ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    else:
        ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


In [ ]:
ds_train = make_ds(train_files, train_labels, training=True)
ds_val = make_ds(val_files, val_labels, training=False)
ds_test = make_ds(test_files, test_labels, training=False)
print("Datasets ready")


In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

model.summary()


In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_accuracy", factor=0.5, patience=2),
]

history = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=8,
    callbacks=callbacks,
)


In [ ]:
# Fine-tuning
base_model.trainable = True

# Freeze most layers; keep the top trainable.
for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

ft_history = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=6,
    callbacks=callbacks,
)


In [ ]:
# Final evaluation + save
test_loss, test_acc = model.evaluate(ds_test)
print(f"Test accuracy: {test_acc * 100:.2f}%")

model.save(MODEL_PATH)
print("Saved model to:", MODEL_PATH)
